# G7B · Exploración documental en Deepnote

Primer cuaderno de análisis, separado del pipeline operativo. Lee un lote local o, opcionalmente, Azure o MongoDB. No inicia arañas, no publica en MongoDB y no cambia controles de ingesta.

El objetivo es explorar cobertura, integridad y ejemplos. Las alertas son candidatas a revisión, no una calificación automática de utilidad. Esto **no es telemetría ni el cubo de rendimiento** previsto en el plan.

Antes de ejecutar instala `requirements-deepnote.txt` en el entorno del proyecto. El archivo de datos se carga aparte: no está incluido en el cuaderno ni en Git.

In [ ]:
import os, json, gzip, hashlib, random, re
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
from IPython.display import display

# Cambiar a azure o mongo solo después de configurar una conexión de lectura.
FUENTE = os.getenv("G7B_FUENTE", "local")
ARCHIVO = Path(os.getenv("G7B_ARCHIVO", "883618-1-13.jsonl.gz"))
ORIGEN_LOTE = "bronce/zyte/blender_docs/2026-11-01/883618-1-13.jsonl.gz"
BLOB = "plata/aceptados/zyte/blender_docs/2026-11-01/883618-1-13.jsonl.gz"
MAX_BYTES = 50 * 1024 * 1024
MAX_REGISTROS = 10000
SEMILLA = 20261102
SALIDA = Path("resultados_deepnote")


## Lectura y procedencia

El cargador detecta gzip por su contenido: algunas descargas mantienen `.gz` aunque ya están descomprimidas. Los límites evitan cargar accidentalmente un lago completo.

Azure: configurar `G7B_AZURE_LAGO_URL` como variable secreta con acceso de lectura al contenedor. MongoDB: configurar `G7B_MONGO_URI` con un usuario de lectura; se consulta **solo el origen del lote indicado**, no toda la colección. No pegar credenciales en celdas. Los errores de conexión se resumen para no mostrar URLs firmadas.

In [ ]:
def leer_jsonl(datos):
    if len(datos) > MAX_BYTES:
        raise ValueError("Archivo comprimido o plano excede el límite de lectura.")
    if datos[:2] == b"\x1f\x8b":
        import io
        with gzip.GzipFile(fileobj=io.BytesIO(datos)) as f:
            datos = f.read(MAX_BYTES + 1)
    if len(datos) > MAX_BYTES:
        raise ValueError("Contenido descomprimido excede el límite.")
    registros = []
    for numero, linea in enumerate(datos.decode("utf-8-sig").splitlines(), 1):
        if not linea.strip():
            continue
        try:
            doc = json.loads(linea)
        except json.JSONDecodeError:
            raise ValueError(f"JSON inválido en línea {numero}; no se omitieron registros.") from None
        if not isinstance(doc, dict):
            raise ValueError(f"La línea {numero} no es un objeto.")
        registros.append(doc)
        if len(registros) > MAX_REGISTROS:
            raise ValueError("El lote excede MAX_REGISTROS; no se analizó una fracción silenciosamente.")
    if not registros:
        raise ValueError("No hay registros para analizar.")
    return registros, hashlib.sha256(datos).hexdigest()

def cargar():
    if FUENTE == "local":
        if not ARCHIVO.is_file():
            raise FileNotFoundError("Carga el JSONL en Files y ajusta G7B_ARCHIVO o ARCHIVO.")
        with ARCHIVO.open("rb") as f:
            datos = f.read(MAX_BYTES + 1)
        docs, huella = leer_jsonl(datos)
        return docs, {"fuente": "local", "sha256_jsonl": huella}
    if FUENTE == "azure":
        from azure.storage.blob import ContainerClient
        url = os.getenv("G7B_AZURE_LAGO_URL")
        if not url:
            raise ValueError("Falta configurar G7B_AZURE_LAGO_URL en las variables secretas.")
        try:
            with ContainerClient.from_container_url(url) as contenedor:
                blob = contenedor.get_blob_client(BLOB)
                if blob.get_blob_properties().size > MAX_BYTES:
                    raise ValueError("Límite de tamaño")
                datos = blob.download_blob(max_concurrency=1).readall()
        except Exception:
            raise RuntimeError("No se pudo leer el blob: revisar acceso de lectura, vigencia y tamaño sin imprimir credenciales.") from None
        docs, huella = leer_jsonl(datos)
        return docs, {"fuente": "azure", "blob": BLOB, "sha256_jsonl": huella}
    if FUENTE == "mongo":
        from pymongo import MongoClient
        uri = os.getenv("G7B_MONGO_URI")
        if not uri:
            raise ValueError("Falta configurar G7B_MONGO_URI en las variables secretas.")
        campos = {"_id": 0, "url": 1, "titulo": 1, "texto": 1, "fuente": 1,
                  "licencia": 1, "modo": 1, "metadatos": 1, "origen": 1,
                  "puntajes": 1, "preguntas": 1, "hash_contenido": 1}
        try:
            with MongoClient(uri, serverSelectionTimeoutMS=15000, socketTimeoutMS=30000) as cliente:
                coleccion = cliente["conocimiento_3d"]["documentos"]
                docs = list(coleccion.find({"origen": ORIGEN_LOTE}, campos)
                            .sort("url", 1).limit(MAX_REGISTROS + 1).max_time_ms(30000))
        except Exception:
            raise RuntimeError("No se pudo consultar MongoDB: revisar conexión de lectura y permisos sin imprimir credenciales.") from None
        if not docs or len(docs) > MAX_REGISTROS:
            raise ValueError("Consulta vacía o mayor que MAX_REGISTROS; comprobar origen y límite.")
        return docs, {"fuente": "mongo", "origen": ORIGEN_LOTE,
                      "nota": "Instantánea de consulta; no certifica igualdad byte a byte con Plata."}
    raise ValueError("FUENTE debe ser local, azure o mongo.")

documentos, procedencia = cargar()
print(f"Registros leídos: {len(documentos):,}")
display(procedencia)


## Tabla de análisis

Los documentos de Plata y MongoDB guardan los puntajes en posiciones distintas; se normalizan sin inventar el estado del embudo cuando falta. La columna de palabras se calcula sobre el texto recibido y puede diferir del conteo original, que también utiliza el título.

In [ ]:
def mapa(valor):
    return valor if isinstance(valor, dict) else {}

def normalizar(doc, numero):
    meta, embudo = mapa(doc.get("metadatos")), mapa(doc.get("embudo"))
    puntajes = mapa(embudo.get("puntajes", doc.get("puntajes")))
    texto = doc.get("texto") if isinstance(doc.get("texto"), str) else ""
    url = doc.get("url") if isinstance(doc.get("url"), str) else ""
    bloques = re.split(r"\n\s*\n", texto)
    vacios = sum(bool(re.match(r"^#{1,6} ", x)) and bool(re.match(r"^#{1,6} ", y))
                 for x, y in zip(bloques, bloques[1:]))
    version = str(meta.get("version") or "sin_version")
    return {"registro": numero, "url": url, "titulo": doc.get("titulo") or "",
            "texto": texto, "fuente": doc.get("fuente") or "", "version": version,
            "seccion": meta.get("seccion") or "sin_seccion", "licencia": doc.get("licencia") or "",
            "origen": doc.get("origen") or "", "estado": embudo.get("estado", "no_disponible"),
            "valor": puntajes.get("valor"), "palabras_texto": len(texto.split()),
            "encabezados_consecutivos": vacios,
            "icono_titulo": "\uf0c1" in (doc.get("titulo") or ""),
            "url_version_coincide": version != "sin_version" and f"/{version}/" in url,
            "texto_sha256": hashlib.sha256(texto.encode()).hexdigest()}

df = pd.DataFrame(normalizar(d, i) for i, d in enumerate(documentos, 1))
metricas = {"registros": len(df), "urls_duplicadas_adicionales": int(df.url.duplicated().sum()),
            "textos_duplicados_adicionales": int(df.texto_sha256.duplicated().sum()),
            "valor_cero": int(df.valor.eq(0).sum()),
            "puntaje_no_disponible": int(df.valor.isna().sum()),
            "candidatos_encabezados_vacios": int(df.encabezados_consecutivos.ge(3).sum()),
            "titulos_con_icono": int(df.icono_titulo.sum())}
display(pd.Series(metricas, name="Resultado"))
display(pd.DataFrame({"campo": ["url", "titulo", "texto", "licencia", "origen"],
                      "faltantes": [int(df[c].eq("").sum()) for c in ["url", "titulo", "texto", "licencia", "origen"]]}))


## Cobertura y consulta SQL

Esta tabla DuckDB sirve para explorar documentos; no se presenta como el cubo de mediciones del plan. Las versiones son metadatos declarados, no una comprobación de vigencia del manual.

In [ ]:
conexion = duckdb.connect(":memory:")
conexion.register("documentos", df)
display(conexion.sql("SELECT version, count(*) AS documentos FROM documentos GROUP BY version ORDER BY version").df())
display(conexion.sql("SELECT seccion, count(*) AS documentos FROM documentos GROUP BY seccion ORDER BY documentos DESC").df())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
df.version.value_counts().sort_index().plot.bar(ax=axes[0], title="Documentos por versión")
df.seccion.value_counts().sort_values().plot.barh(ax=axes[1], title="Cobertura por sección")
fig.tight_layout()
plt.show()


## Casos para revisión humana

Las alertas no implican rechazo: una sección breve puede ser correcta y los encabezados consecutivos requieren inspección. La muestra siguiente es reproducible dentro del mismo lote, ordenada por URL antes del sorteo; **no es la misma muestra histórica de la auditoría**, que usó otro algoritmo.

In [ ]:
columnas = ["registro", "titulo", "version", "url", "encabezados_consecutivos"]
display(df.loc[df.encabezados_consecutivos.ge(3), columnas])
indices = list(df.sort_values("url").index)
muestra = df.loc[random.Random(SEMILLA).sample(indices, min(20, len(indices)))].copy()
muestra["dictamen_humano"] = "pendiente"
muestra["observacion_humana"] = ""
display(muestra[["registro", "titulo", "version", "url", "dictamen_humano"]])

# Búsqueda literal local; no es búsqueda semántica ni generación de respuestas.
BUSCAR = "Simplify"
VERSION = "4.5"  # Usar "" para todas.
seleccion = df[(df.titulo.str.contains(BUSCAR, case=False, regex=False)
                | df.texto.str.contains(BUSCAR, case=False, regex=False))
               & ((df.version == VERSION) if VERSION else True)]
display(seleccion[["registro", "titulo", "version", "url"]].head(20))
if not seleccion.empty:
    print(seleccion.iloc[0].texto[:8000])


## Exportaciones locales del proyecto

Se guardan métricas, una plantilla de revisión y Parquet de metadatos (sin textos). La plantilla CSV neutraliza prefijos de fórmula para abrirla en hojas de cálculo. No se sube nada a Git ni a Azure/MongoDB. Las salidas del cuaderno pueden contener documentación: revisarlas antes de compartir el proyecto.

In [ ]:
SALIDA.mkdir(parents=True, exist_ok=True)
(SALIDA / "metricas.json").write_text(json.dumps({"procedencia": procedencia, "metricas": metricas}, ensure_ascii=False, indent=2), encoding="utf-8")
revision = muestra[["registro", "titulo", "version", "url", "dictamen_humano", "observacion_humana"]].copy()
def seguro_csv(x):
    return "'" + x if isinstance(x, str) and x.lstrip().startswith(("=", "+", "-", "@")) else x
for columna in revision.select_dtypes(include="object"):
    revision[columna] = revision[columna].map(seguro_csv)
revision.to_csv(SALIDA / "revision_humana.csv", index=False, encoding="utf-8-sig")
df.drop(columns=["texto"]).to_parquet(SALIDA / "documentos_metadatos.parquet", index=False)
print("Exportaciones locales listas en resultados_deepnote/.")
conexion.close()


## Cómo interpretar los resultados

- Valor cero no equivale a basura: el catálogo del manual permite ese umbral.
- Cero duplicados exactos no descarta repeticiones internas o similitudes entre versiones.
- Una URL oficial declarada no certifica fidelidad al original.
- El lote aceptado no permite evaluar falsos rechazos sin leer también los otros lotes.
- Próximo paso del plan: importar mediciones reales y Open Data con sus contratos; este cuaderno no simula esos datos.

Referencias: [plan G7B](https://github.com/Maximiliano-cabello-mata/amatista-g7b-etl/blob/main/docs/PLAN.md), [importar Jupyter](https://deepnote.com/docs/importing-and-exporting-jupyter-notebooks), [variables secretas](https://deepnote.com/docs/environment-variables).